# SumIterative, RecursiveSum e RecursiveSumLinear (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_java.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `SumIterative`, `RecursiveSum` e `RecursiveSumLinear`: gli algoritmi in Java, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursiveSum` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Somma di un array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: un ciclo che accumula le somme parziali. $S(n)=3n+4$, memoria extra $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide et impera sulle due metà (caso base $l=r$). $2n-1$ chiamate, $S(n)=7n-5$, profondità $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decremento di uno: $S(n)=3n-1$, profondità $n$.

*Input*: un array di $n\ge1$ interi (per `RecursiveSum`, due indici $l\le r$). *Output*: la somma degli elementi. I contatori `steps` (righe di pseudocodice eseguite), `calls` e `maxDepth` (chiamate contemporaneamente attive) misurano le grandezze analizzate nel capitolo; in `RecursiveSum` e `RecursiveSumLinear` la `return` è unica per poter aggiornare la profondità.

Sull'array dell'esempio $a=[9,8,12,3,5,14,6]$ ($n=7$) la somma è $57$; `RecursiveSum` fa $13$ chiamate e $44$ passi, con profondità $4$. Poi si esegue la traccia della tabella del capitolo su $a=[9,8,12,3,5]$: la colonna *stack* riporta le chiamate attive.

In [2]:
%%writefile Sums.java
import java.util.*;

public class Sums {
    static long steps = 0;   // passi eseguiti (righe di pseudocodice)
    static int calls = 0;    // chiamate eseguite
    static int depth = 0;    // chiamate attive in questo momento
    static int maxDepth = 0; // profondità massima della ricorsione

    static void reset() {
        steps = 0;
        calls = 0;
        depth = 0;
        maxDepth = 0;
    }

    // Somma iterativa di a[0..n-1]
    static int sumIterative(int[] a, int n) {
        int s = 0;
        int i = 0;
        steps += 2;                 // righe 1-2: s := 0, i := 1
        while (i < n) {
            steps += 3;             // test vero + due righe del corpo
            s = s + a[i];
            i = i + 1;
        }
        steps += 2;                 // test falso + return
        return s;
    }

    // Somma di a[l..r] (indici da 0), l <= r, per divide et impera
    static int recursiveSum(int[] a, int l, int r) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // riga 1: if l = r
        int result;
        if (l == r) {
            steps++;                // riga 2: return a[l] (caso base)
            result = a[l];
        } else {
            steps += 4;             // righe 3-6: m, due chiamate, return
            int m = l + (r - l) / 2;                // divide
            int s1 = recursiveSum(a, l, m);         // conquer (metà sinistra)
            int s2 = recursiveSum(a, m + 1, r);     // conquer (metà destra)
            result = s1 + s2;                       // combine
        }
        depth--;
        return result;
    }

    // Somma di a[0..n-1] per decremento di uno, n >= 1
    static int recursiveSumLinear(int[] a, int n) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // riga 1: if n = 1
        int result;
        if (n == 1) {
            steps++;                // riga 2: return a[1]
            result = a[0];
        } else {
            steps += 2;             // righe 3-4: chiamata e return
            int s = recursiveSumLinear(a, n - 1);
            result = s + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- traccia di RecursiveSum ----
    static int row = 0;
    static int maxStack = 0;

    static void emit(ArrayList<String> stack, String action) {
        row++;
        if (stack.size() > maxStack) maxStack = stack.size();
        System.out.printf("%2d | %s | %s%n", row, String.join(",", stack), action);
    }

    static String p(int l, int r) { return "(" + l + "," + r + ")"; }

    static int recursiveSumTrace(int[] a, int l, int r, ArrayList<String> stack) {
        stack.add(p(l, r));
        if (l == r) {
            emit(stack, "l=r => return a[" + l + "]=" + a[l]);
            stack.remove(stack.size() - 1);
            return a[l];
        }
        int m = l + (r - l) / 2;
        emit(stack, "l!=r, m=" + l + "+(" + r + "-" + l + ")/2=" + m + "; call " + p(l, m));
        int s1 = recursiveSumTrace(a, l, m, stack);
        emit(stack, "s1=" + s1 + "; call " + p(m + 1, r));
        int s2 = recursiveSumTrace(a, m + 1, r, stack);
        emit(stack, "s2=" + s2 + "; return " + s1 + "+" + s2 + "=" + (s1 + s2));
        stack.remove(stack.size() - 1);
        return s1 + s2;
    }

    public static void main(String[] args) {
        int[] a = {9, 8, 12, 3, 5, 14, 6};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        reset();
        int s = sumIterative(a, n);
        System.out.println("SumIterative       = " + s + "   steps=" + steps + " (3n+4=" + (3 * n + 4) + ")");
        reset();
        s = recursiveSum(a, 0, n - 1);
        System.out.println("RecursiveSum       = " + s + "   calls=" + calls + " (2n-1=" + (2 * n - 1) + ")  steps=" + steps + " (7n-5=" + (7 * n - 5) + ")  maxDepth=" + maxDepth);
        reset();
        s = recursiveSumLinear(a, n);
        System.out.println("RecursiveSumLinear = " + s + "   calls=" + calls + "  steps=" + steps + " (3n-1=" + (3 * n - 1) + ")  maxDepth=" + maxDepth);

        int[] b = {9, 8, 12, 3, 5};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   RecursiveSum(a, 0, 4)");
        System.out.println(" # | stack | azione");
        int res = recursiveSumTrace(b, 0, 4, new ArrayList<>());
        System.out.println("risultato " + res + ", righe " + row + ", massimo numero di chiamate attive " + maxStack);
    }
}

Writing Sums.java


In [3]:
!javac Sums.java && java Sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | azione
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Prova tu.** Cambia l'array `a` (anche la lunghezza) e confronta passi, chiamate e profondità: solo `RecursiveSumLinear` ha profondità $n$.

In [4]:
%%writefile ProvaSums.java
import java.util.*;

public class ProvaSums {
    public static void main(String[] args) {
        // Cambia l'array: confronta passi, chiamate e profondità dei tre algoritmi.
        int[] a = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
        int n = a.length;
        Sums.reset(); int s = Sums.sumIterative(a, n);
        System.out.println(s + " " + Sums.steps);
        Sums.reset(); s = Sums.recursiveSum(a, 0, n - 1);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
        Sums.reset(); s = Sums.recursiveSumLinear(a, n);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
    }
}

Writing ProvaSums.java


In [5]:
!javac ProvaSums.java && java ProvaSums

136 52
136 107 31 5
136 47 16 16


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).